# Silver Layer — Clean & Conform Media Data
Dedupe, cast types. Keeps viewing label is_completed; FE drops post-view leakage.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_date, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean content catalog
df_c = spark.read.format('delta').table('bronze_content')
w = Window.partitionBy('content_id').orderBy(col('ingestion_timestamp').desc())
df_c = (
    df_c.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('release_year', col('release_year').cast('int'))
    .withColumn('duration_mins', col('duration_mins').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_c.write.mode('overwrite').format('delta').saveAsTable('silver_content')
print(f'silver_content: {df_c.count()} rows')

In [ ]:
# Clean subscribers
df_s = spark.read.format('delta').table('bronze_subscribers')
w2 = Window.partitionBy('subscriber_id').orderBy(col('ingestion_timestamp').desc())
df_s = (
    df_s.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('monthly_fee', col('monthly_fee').cast('double'))
    .withColumn('is_churned', col('is_churned').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_s.write.mode('overwrite').format('delta').saveAsTable('silver_subscribers')
print(f'silver_subscribers: {df_s.count()} rows')

In [ ]:
# Clean viewing history (keep label is_completed)
df_v = spark.read.format('delta').table('bronze_viewing')
w3 = Window.partitionBy('view_id').orderBy(col('ingestion_timestamp').desc())
df_v = (
    df_v.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('view_date', to_date('view_date'))
    .withColumn('view_hour', col('view_hour').cast('int'))
    .withColumn('watch_duration_mins', col('watch_duration_mins').cast('double'))
    .withColumn('is_completed', col('is_completed').cast('int'))
    .withColumn('rating', col('rating').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_v.write.mode('overwrite').format('delta').saveAsTable('silver_viewing')
print(f'silver_viewing: {df_v.count()} rows')

In [ ]:
# Clean ad impressions
df_a = spark.read.format('delta').table('bronze_ad_impressions')
df_a = (
    df_a
    .withColumn('ad_date', to_date('ad_date'))
    .withColumn('impressions', col('impressions').cast('int'))
    .withColumn('clicks', col('clicks').cast('int'))
    .withColumn('revenue_usd', col('revenue_usd').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_a.write.mode('overwrite').format('delta').saveAsTable('silver_ad_impressions')
print(f'silver_ad_impressions: {df_a.count()} rows')